# Model Evaluation

Evaluate model performance:
1. Calculate metrics (accuracy, precision, recall, F1)
2. Confusion matrix
3. Classification report
4. Visualize results
5. Interpret findings

## Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Add project to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from src import config
from src.data_preprocessing import preprocess_data
from src.evaluate import (
    calculate_metrics, print_metrics, get_confusion_matrix,
    print_confusion_matrix, evaluate_model
)

print("✓ Imports complete")

## Step 1: Load Data and Model

In [ ]:
# Preprocess data
X_train, X_test, y_train, y_test = preprocess_data(project_root / config.RAW_DATA_PATH)

# Load trained model
model = joblib.load(project_root / config.MODEL_PATH)

print(f"✓ Data loaded and model loaded")
print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## Step 2: Make Predictions

In [ ]:
# Make predictions on train and test sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Get prediction probabilities for test set
y_test_proba = model.predict_proba(X_test)

print("✓ Predictions made")
print(f"\nTraining predictions shape: {y_train_pred.shape}")
print(f"Test predictions shape: {y_test_pred.shape}")
print(f"Test probabilities shape: {y_test_proba.shape}")

## Step 3: Calculate Metrics

In [ ]:
# Calculate metrics for both train and test
train_metrics = calculate_metrics(y_train, y_train_pred)
test_metrics = calculate_metrics(y_test, y_test_pred)

# Display metrics
print_metrics(train_metrics, "Training Set")
print_metrics(test_metrics, "Test Set")

# Compare train vs test
print("\nTrain vs Test Comparison:")
print("-" * 40)
for metric in train_metrics.keys():
    train_val = train_metrics[metric]
    test_val = test_metrics[metric]
    diff = train_val - test_val
    print(f"{metric:12} | Train: {train_val:.4f} | Test: {test_val:.4f} | Diff: {diff:+.4f}")

## Step 4: Confusion Matrix

## Step 5: Visualize Confusion Matrices

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training confusion matrix
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['No Flood', 'Flood'], yticklabels=['No Flood', 'Flood'],
            cbar=False)
axes[0].set_title('Training Set Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Test confusion matrix
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['No Flood', 'Flood'], yticklabels=['No Flood', 'Flood'],
            cbar=False)
axes[1].set_title('Test Set Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("✓ Confusion matrix visualization complete!")

## Step 6: Classification Report

In [ ]:
print("\nDetailed Classification Report (Test Set):")
print("=" * 50)
print(classification_report(y_test, y_test_pred, target_names=['No Flood', 'Flood']))

## Step 7: Metrics Visualization

In [ ]:
# Create metrics comparison visualization
metrics_names = list(test_metrics.keys())
train_values = [train_metrics[m] for m in metrics_names]
test_values = [test_metrics[m] for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, train_values, width, label='Train', alpha=0.8)
bars2 = ax.bar(x + width/2, test_values, width, label='Test', alpha=0.8)

ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('Model Performance Metrics: Train vs Test')
ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in metrics_names])
ax.legend()
ax.set_ylim([0, 1.1])
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("✓ Metrics visualization complete!")

## Step 8: ROC Curve (Optional)

## Step 9: Model Interpretation

In [ ]:
# Interpretation summary
interpretation = f"""
## Model Performance Interpretation

### Overall Performance:
- **Test Accuracy**: {test_metrics['accuracy']:.2%}
  - Correct predictions out of total predictions
  
- **Precision**: {test_metrics['precision']:.2%}
  - Of all predicted floods, how many were correct
  - Important to avoid false alarms
  
- **Recall**: {test_metrics['recall']:.2%}
  - Of all actual floods, how many did we catch
  - Important to not miss real flood events
  
- **F1-Score**: {test_metrics['f1']:.4f}
  - Balance between precision and recall

### Overfitting Check:
- Training Accuracy: {train_metrics['accuracy']:.2%}
- Test Accuracy: {test_metrics['accuracy']:.2%}
- Difference: {(train_metrics['accuracy'] - test_metrics['accuracy']):.2%}

{'✓ Good generalization!' if (train_metrics['accuracy'] - test_metrics['accuracy']) < 0.1 else '⚠ Possible overfitting'}

### Confusion Matrix Analysis:
- True Negatives (correct no-flood): {cm_test[0,0]}
- False Positives (false alarms): {cm_test[0,1]}
- False Negatives (missed floods): {cm_test[1,0]}
- True Positives (correct flood): {cm_test[1,1]}
"""

from IPython.display import Markdown, display
display(Markdown(interpretation))

## Summary

✓ Model evaluation complete!

### Key Metrics:
- Accuracy: {{test_metrics['accuracy']:.4f}}
- Precision: {{test_metrics['precision']:.4f}}
- Recall: {{test_metrics['recall']:.4f}}
- F1-Score: {{test_metrics['f1']:.4f}}

### Next Steps:
Continue to `06_making_predictions.ipynb` to make predictions on new data.